# extract_e23
Phase 3 Stage B2 Worker 2 (`extract_e23`): L4 GPU swapped-image midpoint `image_caption` extraction (`last_caption` and `caption_mean` on the same single forward via `src.stage2.evidence.extract_with_evidence`) for all deterministic opposite-label `SCREEN` pairs in `phase3_gpu_plan["e23_plan"]` plus the 8 authentic parity anchor rows (`E35` dropped; `CONFIRM` strictly closed). Run cells 2, 3, and 4 in order. Configure only cell 3. Use a Colab L4 GPU runtime with `requirements-l4.txt`.

In [ ]:
import json, os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned L4 requirements.')
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Research-phase3')
CODE_REF = 'phase3'  # Set to the approved Phase 3 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-l4.txt')], check=True)
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Colab L4 CUDA runtime with bfloat16 support is required.')
gpu_name = torch.cuda.get_device_name(0)
if 'L4' not in gpu_name:
    raise RuntimeError(f'Expected NVIDIA L4 GPU; got {gpu_name!r}')
print('Fixed code commit:', CODE_COMMIT)
print('GPU:', gpu_name)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Phase 3 Stage B2 E23 extraction settings (edit only this cell).
MODEL = 'qwen'  # 'qwen' (depth 18) or 'gemma' (depth 17; requires HF_TOKEN in Colab Secrets).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
PLAN_REF = {
    'path': 'phase3/plans/20261120T060305Z-11b534d878f84475a50b561efbd06650/_COMPLETE.json',
    'sha256': '3603f36631ec8165445613e08caf2a3b74bd7626f84f79fdde59fc34f0088b1c',
}
# Optional explicit chunk resume map ({0: {'path': '...', 'sha256': '...'}}) or None.
RESUME_REFS = None


In [ ]:
import re
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f'Invalid MODEL selector: {MODEL!r}')
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Mounted PHASE1_ROOT not found: {PHASE1_ROOT}')
if PLAN_REF is not None:
    manifest = PHASE1_ROOT / PLAN_REF['path']
    if not manifest.is_file():
        raise RuntimeError(f'Missing mounted PLAN_REF manifest: {manifest}')
run_env = os.environ.copy()
if MODEL == 'gemma':
    hf_token = run_env.get('HF_TOKEN')
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError('Gemma requires HF_TOKEN in Colab Secrets (userdata) or environment.')
    run_env['HF_TOKEN'] = hf_token
def redact(text):
    t = run_env.get('HF_TOKEN')
    if t:
        text = text.replace(t, '[REDACTED]')
    return re.sub(r'hf_[A-Za-z0-9]+', '[REDACTED]', text)
command = [
    sys.executable, '-u', '-m', 'src.phase3.extract_e23',
    '--model', MODEL,
    '--phase1-root', str(PHASE1_ROOT),
    '--storage-scope', 'persistent-drive',
]
if PLAN_REF is not None:
    command.extend(['--plan', json.dumps(PLAN_REF) if isinstance(PLAN_REF, dict) else str(PLAN_REF)])
if RESUME_REFS is not None:
    command.extend(['--resume-refs', json.dumps(RESUME_REFS)])
process = subprocess.Popen(
    command, cwd=REPO, env=run_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
last_result = None
for line in process.stdout:
    safe_line = redact(line)
    print(safe_line, end='', flush=True)
    try:
        payload = json.loads(safe_line)
        if isinstance(payload, dict):
            if payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
            if payload.get('status') == 'passed' and 'release_ref' in payload:
                last_result = payload
    except json.JSONDecodeError:
        pass
code = process.wait()
if code != 0:
    detail = '; '.join(redact(str(b)) for b in blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E23 GPU extraction failed ({code}): {detail}')
if last_result is not None:
    print('\n=== AUTHORITATIVE E23 RELEASE REFERENCE ===')
    print(json.dumps(last_result['release_ref'], indent=2))
